In [ ]:
import os
import sys
import importlib.util
from pathlib import Path
import numpy as np
from dataio.get_input import Benchmark, ModalitySeries, NativeSeries


# OlmoEarth v1.1-Base, frozen encoder (768-d embeddings)

This notebook follows `src/models/olmoearth.py` and shows how the wrapper adapts benchmark Sentinel-2 series into the OlmoEarth sample format.

Key properties:
- Uses Sentinel-2 only.
- Expects 12 OlmoEarth channels; benchmark-missing B01 and B09 are filled and masked.
- Builds images, masks, and timestamps before calling the pretrained encoder.
- Pools the encoder output into one 768-dimensional embedding per parcel.

## What OlmoEarth expects as input

| Tensor | Shape | Meaning |
| --- | --- | --- |
| `images` | `(N, 1, 1, T, 12)` | Sentinel-2 series in OlmoEarth channel order |
| `masks` | `(N, 1, 1, T, 1)` | v1.1 all-band token masks |
| `timestamps` | `(N, T, 3)` | day, zero-indexed month, year for every observation |
| output embedding | `(N, 768)` | pooled frozen representation |

OlmoEarth has its own package dependency. The conversion cell below reports that dependency clearly if it is not installed in the active environment.

In [ ]:

REPO = Path.cwd()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))


def load(name, rel):
    spec = importlib.util.spec_from_file_location(name, REPO / rel)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

In [ ]:
olmoearth = load('olmoearth_mod', 'src/models/olmoearth.py')

print('OlmoEarth bands:', olmoearth.OLMOEARTH_S2_BANDS)
print('embedding dim:', olmoearth.OLMOEARTH_EMBEDDING_DIM)
print('HF repo:', olmoearth.OLMOEARTH_HF_REPO)

In [ ]:
rng = np.random.default_rng(7)
N, T = 4, 18


s2_bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12', 'NDVI']
s1_bands = ['VV', 'VH']
climate_bands = ['temperature', 'precipitation', 'elevation', 'slope']

months = np.arange(T, dtype=np.int64) % 12
doy = np.linspace(15, 350, T).astype(np.float32)
years = np.full(T, 2021, dtype=np.int64)

s2_values = [(rng.random((T, len(s2_bands))) * 5000).astype(np.float32) for _ in range(N)]
s1_values = [rng.normal(-12, 3, size=(T, len(s1_bands))).astype(np.float32) for _ in range(N)]
climate_values = [rng.normal(0, 1, size=(T, len(climate_bands))).astype(np.float32) for _ in range(N)]

native = NativeSeries(
    s2=ModalitySeries(s2_values, [months] * N, [doy] * N, [years] * N, s2_bands),
    s1=ModalitySeries(s1_values, [months] * N, [doy] * N, [years] * N, s1_bands),
    climate=ModalitySeries(climate_values, [months] * N, [doy] * N, [years] * N, climate_bands),
)
bench = Benchmark(
    name='synthetic',
    label_kind='multiclass',
    native=native,
    labels=np.arange(N, dtype=np.int64) % 2,
    groups=np.array(['a', 'a', 'b', 'b'], dtype=object),
    latlon=np.repeat(np.array([[11.0, 79.0]], dtype=np.float32), N, axis=0),
    label_names=['zero', 'one'],
    years=np.full(N, 2021, dtype=np.int64),
)

s2_monthly, _, _, _ = bench.monthly('s2')
s1_monthly, _, _, _ = bench.monthly('s1')
climate_monthly, _, _, _ = bench.monthly('climate')
print('s2 monthly', s2_monthly.shape, bench.s2_bands)
print('s1 monthly', s1_monthly.shape, bench.s1_bands)
print('climate monthly', climate_monthly.shape, bench.climate_bands)
print('latlon', bench.latlon.shape, 'years', bench.years.shape)

## Wrapper mapping

`_bench_to_olmoearth` is the model-specific conversion used internally by `encode`. It relies on `olmoearth_pretrain` for the sample datatypes and normalization helpers.

In [ ]:
enc = olmoearth.OlmoEarthModel(device='cpu')

try:
    images, masks, timestamps = enc._bench_to_olmoearth(bench)
    print('images', images.shape, images.dtype)
    print('masks', masks.shape, masks.dtype)
    print('timestamps', timestamps.shape, timestamps.dtype)
except ModuleNotFoundError as exc:
    print('Skipping conversion because olmoearth_pretrain is not installed:', exc)
except Exception as exc:
    print('Skipping conversion because the wrapper raised:', type(exc).__name__, exc)

## Forward pass -> embedding

The real forward pass requires the OlmoEarth weights. The shared project environment includes the v1.1 package and downloads the public Base weights on first use.

In [ ]:
RUN_REAL_FORWARD = False

enc = olmoearth.OlmoEarthModel(device='cpu')

if not RUN_REAL_FORWARD:
    print('Set RUN_REAL_FORWARD = True to run the frozen encoder.')
else:
    emb = enc.encode(bench)
    print('embedding', emb.shape)
    print('first row, first 8 dims', np.round(emb[0, :8], 4))